
# SoilSenseAI — AI/ML and Deep Learning Development

## 1. Objective

The objective of this stage is to develop reliable machine learning and deep learning capabilities for SoilSenseAI.

The models should support meaningful agricultural prediction tasks using available soil, geographical, and environmental information.

The development process will prioritize reliable data, scientifically meaningful targets, spatial generalization, and explainable results.

## 2. Available Data Sources

### APNA Farm
- Soil pH
- Electrical Conductivity (EC)
- Phosphorus (P)
- Potassium (K)
- Organic Matter (OM)
- Soil Texture

### SoilGrids
- Soil pH
- Soil Organic Carbon (SOC)
- Nitrogen
- Clay
- Sand
- Silt
- Cation Exchange Capacity (CEC)
- Bulk Density
- Coarse Fragments
- Soil Water Content
- Multiple soil-depth intervals

### Potential Future Data
- Laboratory soil measurements
- Weather and rainfall
- Temperature
- Irrigation information
- Field observations
- Crop performance and yield records

## 3. Model Development Principles

1. Identify a meaningful prediction task before selecting an algorithm.
2. Separate input features from target variables.
3. Do not treat modelled estimates as independent laboratory ground truth.
4. Avoid data leakage during training and validation.
5. Prefer geographically independent testing where possible.
6. Compare traditional machine learning with deep learning.
7. Evaluate models using appropriate metrics.
8. Preserve reproducibility and document model limitations.

## 4. Expected Outcome

A validated prediction model, where suitable training data exists, that can be integrated into the SoilSenseAI system.

The model must provide useful information beyond simply reproducing existing soil datasets.



## 2. Candidate AI/ML Prediction Tasks

Before selecting an architecture, potential prediction tasks must be evaluated against the available data.

| Prediction task | Required target | Current feasibility |
|---|---|---|
| Soil pH prediction | Independent pH measurements | Requires laboratory labels for genuine validation |
| Soil nutrient prediction | Laboratory P, K and N measurements | Requires additional verified labels |
| Soil organic matter prediction | Laboratory OM measurements | Requires additional verified labels |
| Soil texture classification | Verified texture classes | Possible exploratory modelling; independent labels needed |
| Soil salinity prediction | Verified EC measurements | Requires additional verified labels |
| Crop suitability prediction | Verified crop suitability or performance | Requires crop requirements and validation data |
| Soil-property spatial estimation | Existing spatial property estimates | Possible, but may reproduce existing source estimates |

### 2.1 Initial Development Decision

The current datasets will first be evaluated for feature and target suitability.

Potential exploratory models may use APNA Farm and SoilGrids as inputs and targets, but such experiments must be clearly labelled as learning from existing spatial estimates.

They must not be presented as independently validated soil predictions.

### 2.2 Preferred Direction

The preferred long-term direction is to train models using independently collected or verified agricultural observations.

This would allow SoilSenseAI to develop predictive capabilities that go beyond reproducing APNA Farm and SoilGrids outputs.


In [4]:

# Purpose: Audit available soil datasets and identify potential model targets.

import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# Define dataset paths
base_path = Path(r"D:\SoilSenseAI")

apna_path = (
    base_path / "data" / "processed" / "ApnaFarmData"
    / "punjab_apna_farm_soil_dataset.csv"
)

soilgrids_path = (
    base_path / "data" / "processed" / "parquet"
    / "punjab_soil_0-5cm.parquet"
)

# Load APNA Farm dataset
apna_df = pd.read_csv(apna_path)

# Read SoilGrids metadata without loading the full dataset
soilgrids_file = pq.ParquetFile(soilgrids_path)

# Display dataset information
print("APNA FARM DATASET")
print("-" * 45)
print("Shape:", apna_df.shape)
print("Available columns:", len(apna_df.columns))

print("\nSOILGRIDS DATASET")
print("-" * 45)
print("Rows:", soilgrids_file.metadata.num_rows)
print("Columns:", soilgrids_file.metadata.num_columns)
print("Schema:")
print(soilgrids_file.schema)

# Identify potential APNA Farm targets
apna_targets = [
    "ec_value_midpoint",
    "k_value_midpoint",
    "om_value_midpoint",
    "p_value_midpoint",
    "ph_value_midpoint",
    "texture_class_label"
]

print("\nAPNA FARM TARGET AVAILABILITY")
print("-" * 45)

for column in apna_targets:
    if column in apna_df.columns:
        available = apna_df[column].notna().sum()
        missing = apna_df[column].isna().sum()

        print(
            f"{column}: "
            f"{available:,} available | "
            f"{missing:,} missing"
        )
    else:
        print(f"{column}: NOT FOUND")

# Identify potential SoilGrids targets
soilgrids_columns = soilgrids_file.schema.names

print("\nSOILGRIDS TARGET AVAILABILITY")
print("-" * 45)

for column in [
    "phh2o", "soc", "nitrogen", "clay",
    "sand", "silt", "cec", "bdod"
]:
    print(
        f"{column}: "
        f"{'AVAILABLE' if column in soilgrids_columns else 'NOT FOUND'}"
    )


APNA FARM DATASET
---------------------------------------------
Shape: (33643, 39)
Available columns: 39

SOILGRIDS DATASET
---------------------------------------------
Rows: 3223093
Columns: 16
Schema:
required group field_id=-1 schema {
  optional float field_id=-1 longitude;
  optional float field_id=-1 latitude;
  optional binary field_id=-1 depth (String);
  optional float field_id=-1 bdod;
  optional float field_id=-1 cec;
  optional float field_id=-1 cfvo;
  optional float field_id=-1 clay;
  optional float field_id=-1 nitrogen;
  optional float field_id=-1 ocd;
  optional float field_id=-1 phh2o;
  optional float field_id=-1 sand;
  optional float field_id=-1 silt;
  optional float field_id=-1 soc;
  optional float field_id=-1 wv0010;
  optional float field_id=-1 wv0033;
  optional float field_id=-1 wv1500;
}


APNA FARM TARGET AVAILABILITY
---------------------------------------------
ec_value_midpoint: 33,548 available | 95 missing
k_value_midpoint: 29,393 available | 4,250 


## 3. Model Feasibility Assessment and Development Decision

### 3.1 Dataset Availability

The dataset audit confirms:

- APNA Farm contains 33,643 unique geographical locations.
- Individual soil properties have between 27,057 and 33,548 available records.
- SoilGrids contains approximately 3.22 million spatial records.
- Both sources provide useful information for spatial soil analysis.

### 3.2 Target Quality Assessment

Despite substantial dataset sizes, neither source currently provides independent laboratory ground-truth labels.

APNA Farm values are digitized map estimates.

SoilGrids values are modelled soil-property estimates.

Training a model against these values may reproduce existing spatial estimates rather than predict independently verified soil conditions.

### 3.3 Development Decision

The initial development will prioritize:

1. Coordinate-based soil-property retrieval.
2. Rule-based soil assessment.
3. Evidence-based soil-risk identification.
4. Preliminary crop suitability screening.

Supervised ML/DL development will proceed when suitable independent target measurements become available.

### 3.4 Future Model Development

Once laboratory records or field measurements are obtained:

- Prepare independent training and testing datasets.
- Establish Random Forest and XGBoost baselines.
- Evaluate an MLP or other appropriate deep learning architecture.
- Use spatial validation.
- Compare performance using suitable evaluation metrics.
- Select the model that demonstrates genuine predictive value.

### 3.5 Final Decision

Model complexity will be determined by data quality and prediction requirements.

A reliable, validated traditional ML model is preferable to an unvalidated deep learning model.
